# MuScripter向け：音声から非ドラムMIDI

あなたの **MuScripter（旋律・和声・ベース担当）** の分業方針に合わせ、非ドラム楽器だけの `music.mid` を作るノートブックです。推論には別プロジェクトの公開モデル **[MuScriptor](https://github.com/muscriptor/muscriptor)** を使います。**あなたの独自MuScripterモデルを動かすものではありません。** 後処理だけでなく、MuScriptor の推論時にドラムを禁止します。

1. Colab の **ランタイム → ランタイムのタイプを変更 → GPU** を選択。
2. [MuScriptor medium のモデルページ](https://huggingface.co/MuScriptor/muscriptor-medium)で条件を確認してアクセスを承認。
3. 自分の Hugging Face 読み取りトークンを用意。トークンはノートブックに書き込まず、入力欄から渡します。

モデル重みは CC BY-NC 4.0 で提供され、モデルページの利用条件が適用されます。初回ダウンロードには時間がかかります。

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU がありません。「ランタイム → ランタイムのタイプを変更 → GPU」を選択してください。')
print('GPU:', torch.cuda.get_device_name(0))
print('空きVRAM: %.1f GiB' % (torch.cuda.mem_get_info()[0] / 1024**3))


## 固定バージョンのライブラリを準備

In [ ]:
%pip -q install "muscriptor==0.3.0" "pretty_midi>=0.2.10,<1"


## Hugging Face にログイン

モデルページへのアクセス承認を先に済ませてください。トークンは出力されません。

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass('Hugging Face read token: ').strip()
if not hf_token:
    raise ValueError('トークンが空です。')
login(token=hf_token, add_to_git_credential=False)
del hf_token


## 楽曲を1つアップロード

自分で使用できる音源を選んでください。推論時間を抑えるため、最初は短い音源で試せます。

In [ ]:
from google.colab import files
from pathlib import Path
import tempfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('音声を1つだけ選んでください。')
name, data = next(iter(uploaded.items()))
suffix = Path(name).suffix.lower()
if suffix not in {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}:
    raise ValueError('WAV / MP3 / FLAC / OGG / M4A を選んでください。')
if len(data) > 100 * 1024**2:
    raise ValueError('100 MiB 以下の音声を選んでください。')
work_dir = Path(tempfile.mkdtemp(prefix='muscripter_', dir='/content'))
audio_path = work_dir / ('input' + suffix)
audio_path.write_bytes(data)
print('入力:', Path(name).name)


## 非ドラム楽器を採譜

初期設定は `medium` モデルです。GPUと空き容量に余裕があれば、対応するモデルページで利用許可を得たうえで `large` に変更できます。量子化はオフにし、ドラムを禁止します。音声からMIDIへの推定なので誤検出は後で修正してください。

In [ ]:
import io
import pretty_midi
from muscriptor import TranscriptionModel
from muscriptor.tokenizer.mt3 import MT3_FULL_PLUS_GROUP_NAMES

MODEL_SIZE = 'medium'  # small / medium / large
if MODEL_SIZE not in {'small', 'medium', 'large'}:
    raise ValueError('MODEL_SIZE は small / medium / large から選んでください。')
non_drum_groups = [name for name in MT3_FULL_PLUS_GROUP_NAMES if name != 'drums']
model = TranscriptionModel.load_model(MODEL_SIZE, device='cuda', dtype='float16')
midi_bytes, beat_grid = model.transcribe_and_postprocess(
    str(audio_path), instruments=non_drum_groups,
    detect_tempo='best-effort', quantize=False,
)
music = pretty_midi.PrettyMIDI(io.BytesIO(midi_bytes))
# MIDI の検査と二重の保証。独立したドラム採譜のトラックは含めない。
music.instruments = [instrument for instrument in music.instruments if not instrument.is_drum]
if not music.instruments or not any(instrument.notes for instrument in music.instruments):
    raise RuntimeError('非ドラムの音符が検出されませんでした。音源を確認してください。')
output_path = work_dir / 'music.mid'
music.write(str(output_path))
print('楽器トラック:', [(i.name, len(i.notes)) for i in music.instruments])
print('BPM 推定:', '成功' if beat_grid is not None else '利用できません')
files.download(str(output_path))
